# 🎹 내 멜로디 + AI — 내 멜로디를 기반으로 새로운 음악을

'자동 채보' 노트북에서 추출한 MIDI 멜로디를 AI에게 들려주면,
여러분의 **멜로디를 기반으로** 완전히 새로운 스타일의 음악을 만들어줍니다.

같은 4마디 멜로디가 재즈가 되기도 하고, 오케스트라가 되기도 합니다.

**도구**: [MusicGen melody](https://github.com/facebookresearch/audiocraft) — 멜로디 컨디셔닝 기능

▶ 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

In [ ]:
%%capture
!apt-get -qq install -y fluidsynth > /dev/null 2>&1
!pip install -q audiocraft midi2audio

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. MIDI 파일 올리기

▶ '자동 채보' 노트북에서 만든 MIDI 파일을 업로드하세요.

MIDI 파일이 없으면 이 셀은 건너뛰고, 다음 셀에서 예시 파일을 사용하세요.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
input_midi = list(uploaded.keys())[0]
print(f'✅ MIDI 파일이 업로드되었습니다: {input_midi}')

---
## 2. (선택) 예시 MIDI 사용

▶ MIDI 파일이 없으면 이 셀을 실행하세요. 예시 멜로디를 다운로드합니다.

In [ ]:
import urllib.request

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
input_midi = 'melody_example.mid'

if not os.path.exists(input_midi):
    try:
        urllib.request.urlretrieve(f'{REPO_URL}/{input_midi}', input_midi)
        print(f'✅ 다운로드 완료: {input_midi}')
    except:
        print(f'⚠️ 다운로드 실패. 위 셀에서 직접 업로드해주세요.')

---
## 3. MIDI를 소리로 변환

MusicGen은 MIDI를 직접 읽지 못하고, **오디오(소리 파형)**를 입력으로 받습니다.
그래서 MIDI를 먼저 소리로 변환하는 과정이 필요합니다.

▶ 아래 셀을 실행하세요.

In [ ]:
from midi2audio import FluidSynth
import IPython.display as ipd

melody_audio = 'melody_audio.wav'

print('🔄 MIDI를 오디오로 변환 중...')
try:
    fs = FluidSynth()
    fs.midi_to_audio(input_midi, melody_audio)
    print('✅ 변환 완료! 이 소리가 AI의 입력이 됩니다.')
    ipd.display(ipd.Audio(melody_audio))
except Exception as e:
    print(f'⚠️ 변환 실패: {e}')
    print('FluidSynth 사운드폰트가 없을 수 있습니다.')

---
## 4. 멜로디 컨디셔닝 모델 불러오기

이번에는 '텍스트 → 음악' 노트북에서 사용한 것과는 다른 모델을 사용합니다.
**musicgen-melody** 모델은 멜로디를 입력받아, 그 위에 새로운 편곡을 입히는 능력이 있습니다.

⏰ 모델 다운로드 ~3.3GB + 로딩에 1~2분 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
from audiocraft.models import MusicGen
import torch
import torchaudio

print('🔄 MusicGen melody 모델을 불러오는 중...')
model = MusicGen.get_pretrained('facebook/musicgen-melody')
model.set_generation_params(duration=8)

print('✅ 모델 로딩 완료!')

# 멜로디 오디오 로드
melody_waveform, melody_sr = torchaudio.load(melody_audio)
print(f'🎵 멜로디 길이: {melody_waveform.shape[1]/melody_sr:.1f}초')

---
## 5. 내 멜로디를 새로운 스타일로

▶ 아래 `prompt`에 원하는 스타일을 적고 실행하세요.
AI가 여러분의 멜로디를 기반으로 그 스타일의 음악을 만들어줍니다.

**프롬프트 예시:**

| 프롬프트 | 결과 |
|---------|------|
| `transform this melody into a warm jazz arrangement` | 재즈 편곡 |
| `orchestral version with strings and woodwinds` | 오케스트라 편곡 |
| `lo-fi hip hop beat built around this piano melody` | 로파이 힙합 |

In [ ]:
import soundfile as sf
import IPython.display as ipd

prompt = "transform this melody into a warm jazz arrangement"  # ← 여기를 수정하세요

print(f'🎵 프롬프트: "{prompt}"')
print('🔄 멜로디를 기반으로 음악을 생성하고 있습니다...')

wav = model.generate_with_chroma(
    descriptions=[prompt],
    melody_wavs=melody_waveform.unsqueeze(0),
    melody_sample_rate=melody_sr,
)

audio_data = wav[0, 0].cpu().numpy()
output_file = 'melody_conditioned.wav'
sf.write(output_file, audio_data, 32000)

print('✅ 생성 완료!')
print('\n🎵 AI가 만든 음악:')
ipd.display(ipd.Audio(audio_data, rate=32000))

print('\n🎵 원본 멜로디 (비교용):')
ipd.display(ipd.Audio(melody_audio))

---
## 6. 같은 멜로디, 다른 스타일

▶ 프롬프트만 바꿔서 다시 실행해보세요.

💡 같은 멜로디인데 프롬프트에 따라 완전히 다른 음악이 됩니다.
재즈가 되기도 하고, 오케스트라가 되기도 합니다.

In [ ]:
prompt = "orchestral version with strings and woodwinds"  # ← 여기를 수정하세요

print(f'🎵 프롬프트: "{prompt}"')
print('🔄 멜로디를 기반으로 음악을 생성하고 있습니다...')

wav = model.generate_with_chroma(
    descriptions=[prompt],
    melody_wavs=melody_waveform.unsqueeze(0),
    melody_sample_rate=melody_sr,
)

audio_data = wav[0, 0].cpu().numpy()
output_file2 = 'melody_conditioned_2.wav'
sf.write(output_file2, audio_data, 32000)

print('✅ 생성 완료!')
ipd.display(ipd.Audio(audio_data, rate=32000))

---
## 7. 결과 다운로드

▶ 마음에 드는 결과물을 다운로드하세요.

생성한 음악을 'AI 오디오리액티브' 노트북의 입력으로 사용할 수도 있습니다.

In [ ]:
from google.colab import files

for f in ['melody_conditioned.wav', 'melody_conditioned_2.wav']:
    if os.path.exists(f):
        print(f'📥 다운로드: {f}')
        files.download(f)